# Step 5 - Lap stats
> Using the lap times found from part 4, determine the max speed, max acceleration, time spent accelerating, and time spent coasting.

In [1]:
import polars as pL
import numpy as np
import matplotlib.pyplot as plt

pd = pL.read_parquet("../software-data.parquet")
pd = pd.fill_null(strategy="forward")

rpm = pd["SME_TRQSPD_Speed"]
time = pd["Time"]

gear_ratio = 12/41
radius = .2                             #in meters

def meters_second(rpm):
    speed_mps = (rpm * gear_ratio * 2 * 3.14159 * radius) / 60  # Convert rpm to m/s
    return speed_mps

speed = meters_second(rpm)
moving = speed > 0.1

In [2]:
def find_periods(condition):
    # turns a True/False mask into lists of start and end times
    # merges gaps under 0.1 s, drops periods 0.7 s or shorter
    total_periods = time.filter(condition)
    gaps = total_periods.diff()
    starts = total_periods.filter(gaps > .1)
    ends = total_periods.shift(1).filter(gaps > .1)

    starts = starts.to_list()
    ends = ends.to_list()
    total_periods = total_periods.to_list()
    starts = [total_periods[0]] + starts
    ends = ends + [total_periods[-1]]

    filtered_starts = []
    filtered_ends = []
    for s, e in zip(starts, ends):
        if e - s > 0.7:
            filtered_starts.append(s)
            filtered_ends.append(e)

    return filtered_starts, filtered_ends

In [5]:
# lap windows from Step 4
lap_mask = (time >= 20) & (time <= 68)
lap1_times = time.filter(lap_mask)
lap1_speed = speed.filter(lap_mask)

lap_mask2 = (time >= 95) & (time <= 133)
lap2_times = time.filter(lap_mask2)
lap2_speed = speed.filter(lap_mask2)

## 1. Max speed

In [4]:
for name, t, v in [("Lap 1", lap1_times, lap1_speed), ("Lap 2", lap2_times, lap2_speed)]:
    i = v.arg_max()
    print(f"{name}: {v.max():.2f} m/s ({v.max() * 3.6:.1f} km/h) at {t[i]:.2f} s")

Lap 1: 10.54 m/s (37.9 km/h) at 24.69 s
Lap 2: 11.68 m/s (42.1 km/h) at 99.05 s


This is **wheel** speed (from motor RPM). Lap 2's peak lands on a wiggle right after a hard launch (likely wheelspin / driveline wind-up). A 0.5 s average gives 10.83 m/s and lag-corrected GPS ~11.0 m/s, so the car's true top speed was probably closer to 11 m/s.

## 2. Max acceleration
Acceleration = dv/dt. Row to row (`.diff()`) gave 92 m/s² (~9.4 g), which is impossible: the speed sensor only updates every 2nd row, so the whole change lands in one 0.012 s step. Using a 0.5 s window (`.diff(42)`, ~84 rows per second) averages that out.

In [ ]:
n = 42      # ~0.5 s
for name, t, v in [("Lap 1", lap1_times, lap1_speed), ("Lap 2", lap2_times, lap2_speed)]:
    acceleration = v.diff(n) / t.diff(n)
    i = acceleration.arg_max()
    print(f"{name}: {acceleration.max():.2f} m/s^2 ({acceleration.max() / 9.81:.2f} g) at {t[i]:.2f} s")

Window size matters (Lap 1: 9.19 m/s² at 0.25 s, 6.66 at 0.5 s, 4.38 at 1 s) because the hard push is short. The IMU puts the peak at the same moment but lower (~4.5 m/s²), likely wheelspin: motor RPM measures how fast the wheels spin, the IMU how fast the car speeds up.

## 3. Time spent accelerating
**Accelerating = gaining speed** (dv/dt > 0 over 0.5 s), not "pedal > 0" - light pedal can just hold speed. dv/dt is computed on the whole run first, then cut to the laps, so the first 0.5 s of each lap isn't lost.

In [ ]:
def time_in_lap(starts, ends, lap_start, lap_end):
    # clip each period to the lap window, skip ones fully outside, add up the rest
    total = 0
    for s, e in zip(starts, ends):
        s = max(s, lap_start)
        e = min(e, lap_end)
        if e > s:
            print(f"   {s:.2f} - {e:.2f}  ({e - s:.2f} s)")
            total += e - s
    return total

full_acceleration = speed.diff(n) / time.diff(n)
gaining_speed = full_acceleration > 0
gain_starts, gain_ends = find_periods(gaining_speed)

print("Lap 1 time accelerating:")
print(f"   total: {time_in_lap(gain_starts, gain_ends, 20, 68):.2f} s")
print("Lap 2 time accelerating:")
print(f"   total: {time_in_lap(gain_starts, gain_ends, 95, 133):.2f} s")

## 4. Time spent coasting
**Coasting = no throttle (pedal == 0) AND no brake (voltage <= 375) AND moving.** It's about what the driver is doing, so "not gaining speed" isn't part of it - a car rolling downhill with nothing pressed is still coasting.

In [ ]:
brake = pd["ETC_STATUS_BRAKE_SENSE_VOLTAGE"]
pedal = pd["ETC_STATUS_PEDAL_TRAVEL"]

# Step 5 coasting: no throttle, no brake, still moving
no_throttle = pedal == 0
no_brake = brake <= 375
coasting_5 = no_throttle & no_brake & moving

coast_starts, coast_ends = find_periods(coasting_5)

print("Lap 1 time coasting:")
print(f"   total: {time_in_lap(coast_starts, coast_ends, 20, 68):.2f} s")
print("Lap 2 time coasting:")
print(f"   total: {time_in_lap(coast_starts, coast_ends, 95, 133):.2f} s")

## Summary

| | Lap 1 | Lap 2 |
|---|---|---|
| Max speed | 10.54 m/s | 11.68 m/s |
| Max acceleration (0.5 s) | 6.66 m/s² | 7.13 m/s² |
| Time accelerating | 9.90 s | 4.67 s |
| Time coasting | 37.18 s | 32.83 s |

Notes:
- The brake is never pressed in either lap (max voltage 372 < 375). `ETC_STATUS_BRAKELIGHT` reads 1 for 95% of the run, so it wasn't used.
- Lap 2's gap at 103.03 - 103.83 s is the driver tapping the pedal; speed kept dropping, so it's neither accelerating nor coasting.
- Accelerating and coasting overlap slightly (~0.75 s Lap 1, ~0.34 s Lap 2) because dv/dt looks back 0.5 s.